In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
# Define the folder path and the column names
# '/Users/jul/Desktop/uni/Data Analytics/project/PROBE-202411'
# /Users/jul/Desktop/uni/Data Analytics/PROBE-202409
# '/Users/jul/Desktop/uni/Data Analytics/PROBE-202412'
folder_paths = [
    '/Users/jul/Desktop/uni/Data Analytics/project/PROBE-202411'
]
columns = [
    'VehicleID',
    'gpsvalid',
    'lat',
    'lon',
    'timestamp',
    'speed',
    'heading',
    'for_hire_light',
    'engine_acc'
]

In [3]:
# Initialize an empty list to store the dataframes
all_dfs = []

In [4]:

# Loop through all folders
for folder_path in folder_paths:
    # Loop through all files in the directory
    for filename in os.listdir(folder_path):
        # Check if the file is a CSV file
        if filename.endswith('.csv.out'):
            file_path = os.path.join(folder_path, filename)

            # Read the CSV file into a dataframe with the specified column names
            df = pd.read_csv(file_path, names=columns)

            # Append the dataframe to the list
            all_dfs.append(df)

# Concatenate all dataframes in the list into a single dataframe
combined_df = pd.concat(all_dfs, ignore_index=True)

Filter to Just BMR Bangkok

In [5]:
#Define Regions
BKK_REGION_BOUNDS = {
    'min_lat': 13.4,
    'max_lat': 14.2,
    'min_lon': 99.9,
    'max_lon': 101.3
}

Feature Engineering


In [ ]:
# Empty Travel Time Prediction - Data Preparation with Parallel Processing
# Starting from your cleaned taxi data to build inter-zone empty travel time model

import pandas as pd
import numpy as np
import h3
from datetime import timedelta
import gc
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from multiprocessing import Pool, cpu_count
import time

from tqdm import tqdm

# Filter to Bangkok region only
combined_df = combined_df[
    (combined_df['lat'] >= BKK_REGION_BOUNDS['min_lat']) &
    (combined_df['lat'] <= BKK_REGION_BOUNDS['max_lat']) &
    (combined_df['lon'] >= BKK_REGION_BOUNDS['min_lon']) &
    (combined_df['lon'] <= BKK_REGION_BOUNDS['max_lon'])
].reset_index(drop=True)

# Convert timestamp to datetime
combined_df['timestamp'] = pd.to_datetime(combined_df['timestamp'])

# Free memory from original dataframe
gc.collect()

# Check data size and process in chunks if needed
chunk_size = 1_000_000  # 1M points per chunk
print(f"Processing {len(combined_df):,} GPS points on {cpu_count()} cores...")
if len(combined_df) > chunk_size:
    print(f"Large dataset detected. Processing in chunks of {chunk_size:,} points")
    
def get_h3_batch(coords_batch, h3_res=7):
    results = []
    for lat, lon in coords_batch:
        if pd.isna(lat) or pd.isna(lon):
            results.append(None)
        else:
            try:
                if hasattr(h3, 'geo_to_h3'):
                    results.append(h3.geo_to_h3(lat, lon, h3_res))
                else:
                    results.append(h3.latlng_to_cell(lat, lon, h3_res))
            except:
                results.append(None)
    return results

def haversine_distance1(lon1, lat1, lon2, lat2):
    """
    Calculate the great-circle distance between two points 
    on the earth (specified in decimal degrees).
    """
    # convert decimal degrees to radians 
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])

    # haversine formula 
    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a)) 
    r = 6371 # Radius of earth in kilometers.
    return c * r

def assign_h3_zones_parallel(df, lat_col='lat', lon_col='lon', h3_res=7):
    """
    Assign H3 zones to GPS coordinates using parallel processing
    """
    from functools import partial
    
    df = df.copy()
    coords = list(zip(df[lat_col], df[lon_col]))
    
    # Split into batches for parallel processing
    batch_size = len(coords) // cpu_count() + 1
    batches = [coords[i:i+batch_size] for i in range(0, len(coords), batch_size)]
    
    print(f"Processing H3 zones in {len(batches)} batches...")
    
    # Create partial function with h3_res
    h3_func = partial(get_h3_batch, h3_res=h3_res)
    
    # Process in parallel using ThreadPoolExecutor
    with ThreadPoolExecutor(max_workers=cpu_count()) as executor:
        results = list(executor.map(h3_func, batches))
    
    # Flatten results
    h3_zones = [zone for batch_result in results for zone in batch_result]
    df['h3_zone'] = h3_zones
    
    return df

def process_vehicle_trips(vehicle_data_tuple):
    """Process empty trips for a single vehicle"""
    vehicle_id, vehicle_data = vehicle_data_tuple
    vehicle_data = vehicle_data.sort_values('timestamp').reset_index(drop=True)
    
    if len(vehicle_data) < 2:
        return []
        
    empty_trips = []
    
    # Find state changes in for_hire_light
    vehicle_data['prev_for_hire'] = vehicle_data['for_hire_light'].shift(1)
    
    # Find start of empty trips (0 -> 1 transition or first point if already 1)
    empty_starts = vehicle_data[
        ((vehicle_data['for_hire_light'] == 1) & (vehicle_data['prev_for_hire'] == 0)) |
        ((vehicle_data['for_hire_light'] == 1) & (vehicle_data['prev_for_hire'].isna()))
    ].copy()
    
    # Find end of empty trips (1 -> 0 transition)
    empty_ends = vehicle_data[
        (vehicle_data['for_hire_light'] == 0) & (vehicle_data['prev_for_hire'] == 1)
    ].copy()
    
    # Match start and end points
    for _, start_row in empty_starts.iterrows():
        start_time = start_row['timestamp']
        start_zone = start_row['h3_zone']
        
        if pd.isna(start_zone):
            continue
            
        # Find the next end point after this start
        next_ends = empty_ends[empty_ends['timestamp'] > start_time]
        
        if len(next_ends) > 0:
            end_row = next_ends.iloc[0]
            end_time = end_row['timestamp']
            end_zone = end_row['h3_zone']
            
            if pd.isna(end_zone) or start_zone == end_zone:
                continue
                
            # Calculate travel time
            travel_time_minutes = (end_time - start_time).total_seconds() / 60
            
                        # MODIFIED SECTION: Replaced the original simple filter
            # Filter for reasonable travel times (1 to 120 minutes)
            if 1 <= travel_time_minutes <= 120:
                # ADDED: Calculate direct distance between the start and end points
                direct_distance_km = haversine_distance1(
                    start_row['lon'], start_row['lat'],
                    end_row['lon'], end_row['lat']
                )
                
                # ADDED: Calculate the implied speed in km/h
                implied_speed_kmh = direct_distance_km / (travel_time_minutes / 60)

            
                # ADDED: New filter to remove "ghost trips" (e.g., waiting, breaks)
                # A speed less than 3 km/h is slower than walking and is not an active trip.
                if implied_speed_kmh > 5:
                    empty_trips.append({
                        'vehicle_id': vehicle_id,
                        'origin_zone': start_zone,
                        'destination_zone': end_zone,
                        'origin_lat': start_row['lat'],
                        'origin_lon': start_row['lon'],
                        'dest_lat': end_row['lat'],
                        'dest_lon': end_row['lon'],
                        'start_time': start_time,
                        'end_time': end_time,
                        'travel_time_minutes': travel_time_minutes,
                        'hour': end_time.hour,
                        'day_of_week': end_time.dayofweek,
                        'is_weekend': int(end_time.dayofweek >= 5)
                    })
    
    return empty_trips

def extract_empty_trips(df):
    """
    Extract complete empty travel segments from dropoff to pickup using parallel processing
    """
    print("Extracting complete empty travel segments...")
    
    # Sort by vehicle and timestamp
    df = df.sort_values(['VehicleID', 'timestamp']).reset_index(drop=True)
    
    # Assign H3 zones using parallel processing
    df = assign_h3_zones_parallel(df)
    
    # Prepare data for parallel processing by vehicle
    print(f"Processing {df['VehicleID'].nunique()} vehicles in parallel...")
    vehicle_groups = list(df.groupby('VehicleID'))
    
    # Process vehicles in parallel using ThreadPoolExecutor
    with ThreadPoolExecutor(max_workers=cpu_count()) as executor:
        all_vehicle_trips = list(executor.map(process_vehicle_trips, vehicle_groups))
    
    # Flatten results
    empty_trips = [trip for vehicle_trips in all_vehicle_trips for trip in vehicle_trips]
    
    empty_trips_df = pd.DataFrame(empty_trips)
    print(f"Extracted {len(empty_trips_df)} complete empty travel segments")
    return empty_trips_df

def calculate_zone_features(empty_trips_df):
    """
    Calculate additional features for origin-destination pairs
    """
    print("Calculating zone-pair features...")
    
    # Haversine distance between zones
    def haversine_distance(lon1, lat1, lon2, lat2):
        lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
        dlon = lon2 - lon1
        dlat = lat2 - lat1
        a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
        c = 2 * np.arcsin(np.sqrt(a))
        r = 6371  # Earth radius in km
        return c * r
    
    empty_trips_df = empty_trips_df.copy()
    
    # Direct distance between zones
    empty_trips_df['direct_distance_km'] = haversine_distance(
        empty_trips_df['origin_lon'], empty_trips_df['origin_lat'],
        empty_trips_df['dest_lon'], empty_trips_df['dest_lat']
    )
    
    # Direction (bearing) between zones
    def calculate_bearing(lon1, lat1, lon2, lat2):
        lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
        dlon = lon2 - lon1
        y = np.sin(dlon) * np.cos(lat2)
        x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
        bearing = np.arctan2(y, x)
        bearing = np.degrees(bearing)
        return (bearing + 360) % 360
    
    empty_trips_df['bearing_degrees'] = calculate_bearing(
        empty_trips_df['origin_lon'], empty_trips_df['origin_lat'],
        empty_trips_df['dest_lon'], empty_trips_df['dest_lat']
    )
    
    # Cyclical encoding for time features
    empty_trips_df['hour_sin'] = np.sin(2 * np.pi * empty_trips_df['hour'] / 24)
    empty_trips_df['hour_cos'] = np.cos(2 * np.pi * empty_trips_df['hour'] / 24)
    empty_trips_df['dow_sin'] = np.sin(2 * np.pi * empty_trips_df['day_of_week'] / 7)
    empty_trips_df['dow_cos'] = np.cos(2 * np.pi * empty_trips_df['day_of_week'] / 7)
    
    # Direction encoding (North, South, East, West components)
    empty_trips_df['bearing_sin'] = np.sin(np.radians(empty_trips_df['bearing_degrees']))
    empty_trips_df['bearing_cos'] = np.cos(np.radians(empty_trips_df['bearing_degrees']))
    
    return empty_trips_df

def prepare_model_features(empty_trips_df):
    """
    Prepare final feature matrix and target variable for machine learning
    """
    print("Preparing model features...")
    
    # Select and create feature columns
    feature_cols = [
        # Zone identifiers (will be encoded)
        'origin_zone', 'destination_zone',
        
        # Geometric features
        'direct_distance_km', 'bearing_sin', 'bearing_cos',
        
        # Time features
        'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend',
        
        # Traffic features
        'is_morning_rush', 'is_evening_rush', 'is_rush_hour'
    ]
    
    # Create feature matrix
    features_df = empty_trips_df[feature_cols].copy()
    target = empty_trips_df['travel_time_minutes'].copy()
    
    # One-hot encode time periods instead of using in features
    time_period_dummies = pd.get_dummies(
        empty_trips_df['time_period'], 
        prefix='time_period'
    )
    features_df = pd.concat([features_df, time_period_dummies], axis=1)
    
    # Label encode origin and destination zones
    from sklearn.preprocessing import LabelEncoder
    
    # Combine origin and destination zones for consistent encoding
    all_zones = pd.concat([
        empty_trips_df['origin_zone'], 
        empty_trips_df['destination_zone']
    ]).dropna().unique()
    
    zone_encoder = LabelEncoder()
    zone_encoder.fit(all_zones)
    
    features_df['origin_zone_encoded'] = zone_encoder.transform(features_df['origin_zone'])
    features_df['destination_zone_encoded'] = zone_encoder.transform(features_df['destination_zone'])
    
    # Drop original zone columns
    features_df = features_df.drop(['origin_zone', 'destination_zone'], axis=1)
    
    # Add route pair identifier for potential embedding
    features_df['route_pair_id'] = (
        features_df['origin_zone_encoded'] * 10000 + 
        features_df['destination_zone_encoded']
    )
    
    print(f"Feature matrix shape: {features_df.shape}")
    print(f"Target variable shape: {target.shape}")
    print(f"Features: {list(features_df.columns)}")
    
    return features_df, target, zone_encoder

def add_traffic_features(empty_trips_df):
    """
    Add traffic and congestion features
    """
    print("Adding traffic pattern features...")
    
    empty_trips_df = empty_trips_df.copy()
    
    # Rush hour indicators
    empty_trips_df['is_morning_rush'] = empty_trips_df['hour'].isin([7, 8, 9]).astype(int)
    empty_trips_df['is_evening_rush'] = empty_trips_df['hour'].isin([17, 18, 19]).astype(int)
    empty_trips_df['is_rush_hour'] = (
        empty_trips_df['is_morning_rush'] | empty_trips_df['is_evening_rush']
    ).astype(int)
    
    # Time period categories
    def get_time_period(hour):
        if 5 <= hour < 10:
            return 'morning_rush'
        elif 10 <= hour < 16:
            return 'midday'
        elif 16 <= hour < 20:
            return 'evening_rush'
        elif 20 <= hour < 24:
            return 'evening'
        else:
            return 'late_night'
    
    empty_trips_df['time_period'] = empty_trips_df['hour'].apply(get_time_period)
    
    return empty_trips_df

# Main execution function
def build_empty_travel_dataset(df):
    """
    Complete pipeline to build empty travel time prediction dataset
    """
    print("=== Building Empty Travel Time Prediction Dataset ===")
    start_time = time.time()
    
    # Step 1: Extract empty trips
    empty_trips = extract_empty_trips(df)
    
    if len(empty_trips) == 0:
        print("No empty trips found!")
        return None, None, None
    
    # Step 2: Calculate zone features
    empty_trips = calculate_zone_features(empty_trips)

    # Step 3: Add traffic features
    empty_trips = add_traffic_features(empty_trips)
    
    # Step 4: Prepare model features
    X, y, zone_encoder = prepare_model_features(empty_trips)
    
    # Data validation and statistics
    print(f"\nDataset Summary:")
    print(f"- Average distance: {empty_trips['direct_distance_km'].mean():.2f} km")
    print(f"- Distance range: {empty_trips['direct_distance_km'].min():.2f} - {empty_trips['direct_distance_km'].max():.2f} km")
    print(f"- Time range: {y.min():.1f} - {y.max():.1f} minutes")
    print(f"- Total empty trips: {len(X)}")
    print(f"- Unique origin zones: {empty_trips['origin_zone'].nunique()}")
    print(f"- Unique destination zones: {empty_trips['destination_zone'].nunique()}")
    print(f"- Unique route pairs: {empty_trips.groupby(['origin_zone', 'destination_zone']).size().count()}")
    print(f"- Average travel time: {y.mean():.2f} minutes")
    print(f"- Travel time std: {y.std():.2f} minutes")
    
    # Optional: Remove outliers
    distance_q99 = empty_trips['direct_distance_km'].quantile(0.99)
    time_q99 = y.quantile(0.99)
    print(f"- 99th percentile distance: {distance_q99:.2f} km")
    print(f"- 99th percentile time: {time_q99:.1f} minutes")
    
    total_time = time.time() - start_time
    print(f"\n=== Processing completed in {total_time:.1f} seconds ===")
    
    return X, y, empty_trips

# Usage example:
X, y, empty_trips_df = build_empty_travel_dataset(combined_df)

# Save the results
X.to_csv('X_features_inter_zone.csv', index=False)
y.to_csv('y_target_inter_zone.csv', index=False)
empty_trips_df.to_csv('empty_trips_full_data.csv', index=False)

print("Files saved successfully!")
print("- X_features_inter_zone.csv: Feature matrix")
print("- y_target_inter_zone.csv: Target variable (travel times)")
print("- empty_trips_full_data.csv: Complete dataset with all features")

Processing 43,400,146 GPS points on 8 cores...
Large dataset detected. Processing in chunks of 1,000,000 points
=== Building Empty Travel Time Prediction Dataset ===
Extracting complete empty travel segments...
Processing H3 zones in 8 batches...
Processing 3122 vehicles in parallel...
Extracted 354528 complete empty travel segments
Calculating zone-pair features...
Adding traffic pattern features...
Preparing model features...
Feature matrix shape: (354528, 19)
Target variable shape: (354528,)
Features: ['direct_distance_km', 'bearing_sin', 'bearing_cos', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend', 'is_morning_rush', 'is_evening_rush', 'is_rush_hour', 'time_period_evening', 'time_period_evening_rush', 'time_period_late_night', 'time_period_midday', 'time_period_morning_rush', 'origin_zone_encoded', 'destination_zone_encoded', 'route_pair_id']

Dataset Summary:
- Average distance: 8.13 km
- Distance range: 0.09 - 94.55 km
- Time range: 1.0 - 120.0 minutes
- Total empty 

: 